In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("jsonplaceholder-to-iceberg")
    .config("spark.jars.packages",
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1,"
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
            "org.postgresql:postgresql:42.7.4")

    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type", "jdbc")
    .config("spark.sql.catalog.lakehouse.uri", "jdbc:postgresql://postgres:5432/metastore")
    .config("spark.sql.catalog.lakehouse.jdbc.user", "dba_admin")
    .config("spark.sql.catalog.lakehouse.jdbc.password", "Agent9-Backtalk6-Zestfully5-Luxury1-Calculus2")
    .config("spark.sql.catalog.lakehouse.jdbc.driver", "org.postgresql.Driver")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3a://lakehouse/warehouse")
    # .config("spark.sql.catalog.lakehouse.catalog-impl", ...)  <-- REMOVIDO

    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin_minio")
    .config("spark.hadoop.fs.s3a.secret.key", "Grunt9-Relenting6-Hula5-Catcall9-Residue3")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")

    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")

    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Spark version: 3.5.0


In [5]:
import requests
import pandas as pd
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

resp = requests.get("https://jsonplaceholder.typicode.com/posts")
resp.raise_for_status()
posts = resp.json()

# Schema explícito evita inferência inconsistente
schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("id", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("body", StringType(), True),
])

pdf = pd.DataFrame(posts)[["userId", "id", "title", "body"]]
df = spark.createDataFrame(pdf, schema=schema)
df = df.withColumn("data_carga", current_timestamp())

df.printSchema()
df.show(5, truncate=50)

root
 |-- userId: integer (nullable = true)
 |-- id: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- body: string (nullable = true)
 |-- data_carga: timestamp (nullable = false)

+------+---+--------------------------------------------------+--------------------------------------------------+-------------------------+
|userId| id|                                             title|                                              body|               data_carga|
+------+---+--------------------------------------------------+--------------------------------------------------+-------------------------+
|     1|  1|sunt aut facere repellat provident occaecati ex...|quia et suscipit\nsuscipit recusandae consequun...|2026-06-14 02:22:44.11553|
|     1|  2|                                      qui est esse|est rerum tempore vitae\nsequi sint nihil repre...|2026-06-14 02:22:44.11553|
|     1|  3|ea molestias quasi exercitationem repellat qui ...|et iusto sed quo iure\nvoluptatem

In [6]:
spark.sql("CREATE SCHEMA IF NOT EXISTS lakehouse.bronze")

spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.bronze.posts_api (
    userId INT,
    id INT,
    title STRING,
    body STRING,
    data_carga TIMESTAMP
)
USING iceberg
LOCATION 's3a://lakehouse/warehouse/bronze/posts_api'
""")

DataFrame[]

In [7]:
df.createOrReplaceTempView("posts_staging")

spark.sql("""
MERGE INTO lakehouse.bronze.posts_api AS target
USING posts_staging AS source
ON target.id = source.id
WHEN MATCHED THEN UPDATE SET
    target.userId = source.userId,
    target.title = source.title,
    target.body = source.body,
    target.data_carga = source.data_carga
WHEN NOT MATCHED THEN INSERT (userId, id, title, body, data_carga)
VALUES (source.userId, source.id, source.title, source.body, source.data_carga)
""")

DataFrame[]

In [8]:
spark.sql("SELECT * FROM lakehouse.bronze.posts_api ORDER BY id LIMIT 10").show()

+------+---+--------------------+--------------------+--------------------+
|userId| id|               title|                body|          data_carga|
+------+---+--------------------+--------------------+--------------------+
|     1|  1|sunt aut facere r...|quia et suscipit\...|2026-06-14 02:23:...|
|     1|  2|        qui est esse|est rerum tempore...|2026-06-14 02:23:...|
|     1|  3|ea molestias quas...|et iusto sed quo ...|2026-06-14 02:23:...|
|     1|  4|eum et est occaecati|ullam et saepe re...|2026-06-14 02:23:...|
|     1|  5|  nesciunt quas odio|repudiandae venia...|2026-06-14 02:23:...|
|     1|  6|dolorem eum magni...|ut aspernatur cor...|2026-06-14 02:23:...|
|     1|  7|magnam facilis autem|dolore placeat qu...|2026-06-14 02:23:...|
|     1|  8|dolorem dolore es...|dignissimos aperi...|2026-06-14 02:23:...|
|     1|  9|nesciunt iure omn...|consectetur animi...|2026-06-14 02:23:...|
|     1| 10|optio molestias i...|quo et expedita m...|2026-06-14 02:23:...|
+------+---+

In [9]:
spark.sql("SELECT COUNT(*) AS total FROM lakehouse.bronze.posts_api").show()

+-----+
|total|
+-----+
|  100|
+-----+



In [10]:
# Histórico de snapshots (controle de versão do Iceberg)
spark.sql("SELECT * FROM lakehouse.bronze.posts_api.history").show()

+--------------------+-------------------+---------+-------------------+
|     made_current_at|        snapshot_id|parent_id|is_current_ancestor|
+--------------------+-------------------+---------+-------------------+
|2026-06-14 02:23:...|5602706106108548574|     NULL|               true|
+--------------------+-------------------+---------+-------------------+

